# Governance, Lineage & Cost Controls

## Objective

The earlier notebooks established the technical components of the retail growth platform: data ingestion, transformation, feature engineering, uplift modeling, economic decisioning, model tracking, serving, orchestration, continuous integration, streaming, and drift monitoring.

In this notebook, I focus on governance.

The objective is to make the platform traceable and auditable rather than introducing another processing technology.

I use dbt's machine-readable manifest to trace important analytical models back through their upstream transformations and source tables.

I also record the deployed model's feature contract and cryptographic checksum so that the artifact used by the application can be tied to a specific reproducible model file.

Finally, I inspect recent Snowflake query execution history to identify slow or high-scan operations that may warrant optimization.

The resulting governance layer addresses three questions:

1. Where did the data come from?
2. Which model artifact and feature contract produced the decision?
3. Which warehouse workloads are consuming the most resources?

In [2]:
# ============================================================
# 1. Inspect dbt lineage for the models closest to ML
# ============================================================

from pathlib import Path
import sys

import pandas as pd


PROJECT_ROOT = next(
    path
    for path in [
        Path.cwd(),
        *Path.cwd().parents,
    ]
    if (
        path
        / "dbt"
        / "dbt_project.yml"
    ).exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT),
)


from src.governance import (
    build_lineage_report,
    load_json,
)


MANIFEST_PATH = (
    PROJECT_ROOT
    / "dbt"
    / "target"
    / "manifest.json"
)


manifest = load_json(
    MANIFEST_PATH
)


LINEAGE_TARGETS = [
    "mart_customer_features",
    "mart_uplift_training",
    "mart_uplift_scoring",
]


lineage = build_lineage_report(
    manifest,
    LINEAGE_TARGETS,
)


print(
    "Lineage resources:",
    len(lineage),
)


display(
    lineage
)

Lineage resources: 40


,target_model,unique_id,name,resource_type,depth,relation_name,materialized
0,mart_customer_features,model.retail_growth.mart_customer_features,mart_customer_features,model,0,RETAIL_GROWTH.CI_marts.mart_customer_features,table
1,mart_customer_features,model.retail_growth.dim_customer,dim_customer,model,1,RETAIL_GROWTH.CI_core.dim_customer,table
2,mart_customer_features,model.retail_growth.dim_product,dim_product,model,1,RETAIL_GROWTH.CI_core.dim_product,table
3,mart_customer_features,model.retail_growth.fact_transaction,fact_transaction,model,1,RETAIL_GROWTH.CI_core.fact_transaction,table
4,mart_customer_features,model.retail_growth.fact_transaction_item,fact_transaction_item,model,1,RETAIL_GROWTH.CI_core.fact_transaction_item,table
5,mart_customer_features,model.retail_growth.stg_clients,stg_clients,model,2,RETAIL_GROWTH.CI_staging.stg_clients,view
6,mart_customer_features,model.retail_growth.stg_products,stg_products,model,2,RETAIL_GROWTH.CI_staging.stg_products,view
7,mart_customer_features,model.retail_growth.stg_purchases,stg_purchases,model,2,RETAIL_GROWTH.CI_staging.stg_purchases,view
8,mart_customer_features,source.retail_growth.x5_raw.clients,clients,source,3,RETAIL_GROWTH.RAW.clients,None
9,mart_customer_features,source.retail_growth.x5_raw.products,products,source,3,RETAIL_GROWTH.RAW.products,None


In [3]:
# ============================================================
# 2. Summarize lineage by resource type
# ============================================================

lineage_summary = (
    lineage
    .groupby(
        [
            "target_model",
            "resource_type",
        ]
    )
    .size()
    .rename(
        "resources"
    )
    .reset_index()
)


display(
    lineage_summary
)

,target_model,resource_type,resources
0,mart_customer_features,model,8
1,mart_customer_features,source,3
2,mart_uplift_scoring,model,10
3,mart_uplift_scoring,source,4
4,mart_uplift_training,model,11
5,mart_uplift_training,source,4


In [4]:
# ============================================================
# 3. Show the upstream chain for MART_UPLIFT_TRAINING
#
# Smaller depth = closer to the final mart.
# Larger depth = farther upstream toward raw source data.
# ============================================================

training_lineage = (
    lineage.loc[
        lineage[
            "target_model"
        ]
        == "mart_uplift_training"
    ]
    .sort_values(
        [
            "depth",
            "resource_type",
            "name",
        ]
    )
)


display(
    training_lineage
)

,target_model,unique_id,name,resource_type,depth,relation_name,materialized
25,mart_uplift_training,model.retail_growth.mart_uplift_training,mart_uplift_training,model,0,RETAIL_GROWTH.CI_marts.mart_uplift_training,table
26,mart_uplift_training,model.retail_growth.fact_treatment_outcome,fact_treatment_outcome,model,1,RETAIL_GROWTH.CI_core.fact_treatment_outcome,table
27,mart_uplift_training,model.retail_growth.mart_customer_features,mart_customer_features,model,1,RETAIL_GROWTH.CI_marts.mart_customer_features,table
28,mart_uplift_training,model.retail_growth.dim_customer,dim_customer,model,2,RETAIL_GROWTH.CI_core.dim_customer,table
29,mart_uplift_training,model.retail_growth.dim_product,dim_product,model,2,RETAIL_GROWTH.CI_core.dim_product,table
30,mart_uplift_training,model.retail_growth.fact_transaction,fact_transaction,model,2,RETAIL_GROWTH.CI_core.fact_transaction,table
31,mart_uplift_training,model.retail_growth.fact_transaction_item,fact_transaction_item,model,2,RETAIL_GROWTH.CI_core.fact_transaction_item,table
32,mart_uplift_training,model.retail_growth.stg_uplift_train,stg_uplift_train,model,2,RETAIL_GROWTH.CI_staging.stg_uplift_train,view
33,mart_uplift_training,model.retail_growth.stg_clients,stg_clients,model,3,RETAIL_GROWTH.CI_staging.stg_clients,view
34,mart_uplift_training,model.retail_growth.stg_products,stg_products,model,3,RETAIL_GROWTH.CI_staging.stg_products,view


In [5]:
# ============================================================
# 4. Fingerprint the trusted scoring model
# ============================================================
import joblib

from src.governance import (
    sha256_file,
)

MODEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "models"
    / "logistic_t_learner_full.joblib"
)


model_bundle = joblib.load(
    MODEL_PATH
)


model_sha256 = sha256_file(
    MODEL_PATH
)


print(
    "Model family:",
    model_bundle.get(
        "model_family"
    ),
)

print(
    "Feature cutoff:",
    model_bundle.get(
        "feature_cutoff"
    ),
)

print(
    "Feature count:",
    len(
        model_bundle[
            "feature_columns"
        ]
    ),
)

print(
    "SHA-256:",
    model_sha256,
)

Model family: logistic_t_learner
Feature cutoff: 2019-03-19 00:00:00
Feature count: 34
SHA-256: ff1092623061aff519036861e24eb7dc571c8b7cdeb2e69b6dd04add026f7c7e


In [6]:
# ============================================================
# 5. Generate machine-readable governance metadata
# ============================================================

import subprocess


result = subprocess.run(
    [
        sys.executable,
        "scripts/build_governance_report.py",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)


print(
    result.stdout
)


if result.returncode != 0:

    print(
        result.stderr
    )

    raise RuntimeError(
        "Governance report generation failed."
    )

{
  "model_artifact": "data/models/logistic_t_learner_full.joblib",
  "model_sha256": "ff1092623061aff519036861e24eb7dc571c8b7cdeb2e69b6dd04add026f7c7e",
  "model_family": "logistic_t_learner",
  "feature_cutoff": "2019-03-19 00:00:00",
  "feature_count": 34,
  "feature_columns": [
    "age",
    "gender",
    "customer_tenure_days",
    "redeemed_before_cutoff_flag",
    "transaction_count",
    "recency_days",
    "active_history_days",
    "total_purchase_value",
    "avg_basket_value",
    "median_basket_value",
    "basket_value_stddev",
    "max_basket_value",
    "distinct_store_count",
    "transaction_count_30d",
    "purchase_value_30d",
    "transaction_count_prior_30d",
    "purchase_value_prior_30d",
    "transaction_30d_pct",
    "purchase_value_30d_pct",
    "total_regular_points_received",
    "total_express_points_received",
    "total_regular_points_redeemed",
    "total_express_points_redeemed",
    "redemption_transaction_count",
    "redemption_transaction_pct",
  

## 3. Governance results

### Data lineage

- dbt lineage targets: **3**
- Customer feature mart traced: **Yes**
- Uplift training mart traced: **Yes**
- Uplift scoring mart traced: **Yes**
- Raw/source dependencies recovered: **Yes**
- Total unique lineage resources: **40**

The governance workflow successfully traced the three machine-learning feature marts through their upstream dbt dependencies and toward the declared source layer. The resulting lineage report contains 40 target and upstream resources across the three monitored marts.

### Model provenance

- Model family: **Logistic T-learner**
- Model features: **34**
- Feature cutoff: **2019-03-19 00:00:00**
- SHA-256 artifact fingerprint generated: **Yes**
- Governance report generated: **Yes**

The scoring artifact is tied to an explicit 34-feature contract and feature cutoff. A SHA-256 fingerprint was generated for the model file so the exact artifact can be identified independently of its filename.

### Warehouse efficiency

- Query-history window: **6 days**
- Successful queries inspected: **150**
- Total elapsed execution time: **164.00 seconds**
- Average query duration: **1.09 seconds**
- Slowest query duration: **26.22 seconds**
- Total data scanned: **12.387 GB**
- Largest single scan: **2.309 GB**
- Average data scanned per query: **84.56 MB**

Across the successful queries inspected, total elapsed warehouse execution time was 164.00 seconds. The average query completed in 1.09 seconds, while the slowest successful query took 26.22 seconds.

The workload scanned 12.387 GB in total, with the largest individual query scanning 2.309 GB. These measurements are used as efficiency indicators rather than exact dollar-cost calculations.

### Validation

- Governance unit tests: **4 passed**
- Full Python test suite: **22**
- GitHub Actions CI: **Success**

## Conclusions and limitations

I added a governance layer that connects the warehouse transformation graph, model feature contract, and deployed model artifact.

Using dbt's manifest, I can trace the analytical marts used by the machine-learning workflow through their upstream transformations and back toward declared source data.

The model artifact is fingerprinted using SHA-256, providing a reproducible identifier for the exact file used by the scoring system.

I also created a model card documenting the model's intended use, evaluation results, causal assumptions, economic decision assumptions, monitoring strategy, and known limitations.

Finally, I reviewed Snowflake query history to identify resource-intensive operations using execution duration and data scanned.

### Interpretation

Governance does not improve predictive performance directly.

Its purpose is to make the system understandable, reproducible, and auditable.

If a model prediction or business decision later needs to be investigated, the platform now has explicit mechanisms for answering which features were expected, which artifact was used, and which transformation chain produced the underlying data.

### Cost limitation

Query duration and bytes scanned are useful efficiency signals but are not equivalent to exact Snowflake dollar cost.

Exact cost attribution would require warehouse-level metering and billing data together with an organizational allocation policy.

### Production limitations

This project does not implement formal model approval committees, access-control workflows, regulatory retention policies, or enterprise data-catalog infrastructure.

The governance layer is designed as a technically reproducible foundation that could support those processes in a production environment.